# Multi-seed matched sweep — Colab Pro

Runs `scripts/run_matched_sweep.py` across **5 seeds** (42–46) on the matched
population. Each seed changes fold assignment and model initialisation; the
protocol and configs are unchanged.

**Setup:**
1. Runtime → Change runtime type → **A100 GPU** (or L4), **High-RAM**.
2. Run all cells. Data loads from HuggingFace — no manual upload needed.
3. Results save incrementally to Google Drive. Completed seeds survive disconnects.

~15 min/seed on A100, ~30 min/seed on L4, ~50 min/seed on T4.

In [ ]:
# Cell 1: GPU check
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'{gpu} — {mem:.0f} GB')

In [ ]:
# Cell 2: Mount Drive for incremental saves
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT = '/content/drive/MyDrive/bmpb/seeds'
os.makedirs(DRIVE_OUT, exist_ok=True)
print(f'Results will save to {DRIVE_OUT}')

In [ ]:
# Cell 3: Clone repo (agent branch) + install
import os, subprocess, sys, shutil

REPO = '/content/bangla-multimodal-political-stance'
if os.path.exists(REPO):
    shutil.rmtree(REPO)

subprocess.run([
    'git', 'clone', '--depth', '1',
    '-b', 'agent/experiments-grouped-evaluation',
    'https://github.com/kishormorol/bangla-multimodal-political-stance.git',
    REPO
], check=True)
os.chdir(REPO)

!pip install -e . -q 2>&1 | tail -5
!pip install gdown datasets -q 2>&1 | tail -3
!python -c "import bmpb; print('bmpb installed OK')"

In [ ]:
# Cell 4: Build corpus from HuggingFace dataset (same as run_cv_colab.ipynb)
import os, subprocess, sys, time
from pathlib import Path

REPO = '/content/bangla-multimodal-political-stance'
os.chdir(REPO)

corpus_path = Path(REPO, 'data', 'processed', 'corpus.csv')

# Try bmpb data + ingest first
for attempt in range(1, 4):
    print(f'=== bmpb data attempt {attempt}/3 ===')
    subprocess.run([sys.executable, '-m', 'bmpb.cli', 'data'])
    csvs = list(Path(REPO, 'data', 'raw').glob('*.csv'))
    if len(csvs) >= 20:
        print(f'Got {len(csvs)} CSVs.')
        break
    print(f'Got {len(csvs)} CSVs, retrying in 20s...')
    time.sleep(20)

result = subprocess.run([sys.executable, '-m', 'bmpb.cli', 'ingest'],
                        capture_output=True, text=True)
print(result.stdout)

# Fallback: build from HF dataset
if not corpus_path.exists():
    print('Ingest failed — building corpus from HF dataset...')
    from datasets import load_dataset
    import pandas as pd

    ds = load_dataset('kishormorol/BanglaPoliticalStance', split='annotated')
    LABEL_NAMES = {0: 'govt_critique', 1: 'neutral', 2: 'govt_leaning'}

    ds_text = ds.remove_columns(['image'])

    print(f'Building corpus from {len(ds_text)} items (text only)...')
    rows = []
    for idx in range(len(ds_text)):
        item = ds_text[idx]
        rows.append({
            'item_id': item['item_id'],
            'title': item['headline'],
            'text': item['headline'],
            'label': item['label'],
            'label_name': LABEL_NAMES.get(item['label'], 'unknown'),
            'article_label_name': '',
            'image_label_name': '',
            'outlet': item['outlet'],
            'outlet_key': item['outlet'].lower(),
            'date': item['date'],
            'source_url': item['source_url'],
            'image_url': '',
            'image_path': '',
            'image_kind': '',
            'has_image': False,
            'annotator_1': '',
            'annotator_2': '',
            'annotator_3': '',
            'article_label': '',
            'image_label': '',
            'text_level': 'headline',
            'source_index': item['item_id'],
        })
    print(f'  Text corpus: {len(rows)} items')

    # Extract images from Arrow table
    images_dir = Path(REPO, 'data', 'raw', 'processed_images')
    images_dir.mkdir(parents=True, exist_ok=True)
    n_saved = 0
    n_failed = 0

    try:
        import io
        from PIL import Image as PILImage
        arrow_table = ds._data
        img_col = arrow_table.column('image')
        print(f'Extracting images ({len(img_col)} rows)...')

        for idx in range(len(img_col)):
            item_id = rows[idx]['item_id']
            try:
                img_struct = img_col[idx].as_py()
                if img_struct and img_struct.get('bytes'):
                    raw_bytes = img_struct['bytes']
                    pil_img = PILImage.open(io.BytesIO(raw_bytes))
                    img_filename = item_id + '.jpg'
                    img_save_path = images_dir / img_filename
                    pil_img.convert('RGB').save(img_save_path, format='JPEG', quality=95)
                    rows[idx]['image_path'] = 'processed_images/' + img_filename
                    rows[idx]['image_kind'] = 'photo'
                    rows[idx]['has_image'] = True
                    n_saved += 1
            except Exception as e:
                n_failed += 1
                if n_failed <= 5:
                    print(f'  Skipping image for {item_id}: {e}')
        print(f'  Images: {n_saved} saved, {n_failed} failed')
    except Exception as e:
        print(f'Could not extract images: {e}')

    corpus = pd.DataFrame(rows)
    corpus_path.parent.mkdir(parents=True, exist_ok=True)
    corpus.to_csv(corpus_path, index=False)
    print(f'Created corpus.csv: {len(corpus)} items ({n_saved} images)')

# Verify
import pandas as pd
df = pd.read_csv(corpus_path)
print(f'\nCorpus ready: {len(df)} items')
print(f'Labels: {df["label_name"].value_counts().to_dict()}')
n_img = int(df["has_image"].sum())
print(f'Items with images: {n_img}')
if n_img == 0:
    print('WARNING: No images. Multimodal models will fail; text models OK.')

In [ ]:
# Cell 5: Run matched sweep — all 5 seeds
#
# Each seed saves to Drive incrementally. If the runtime disconnects,
# rerun this cell — completed seeds are skipped automatically.

import os, subprocess, sys, json, shutil
from pathlib import Path

os.chdir('/content/bangla-multimodal-political-stance')

SEEDS = [42, 43, 44, 45, 46]
DRIVE_OUT = '/content/drive/MyDrive/bmpb/seeds'

env = {
    **os.environ,
    'PYTHONUNBUFFERED': '1',
    'TOKENIZERS_PARALLELISM': 'false',
}

for seed in SEEDS:
    out_dir = f'{DRIVE_OUT}/seed-{seed}'
    summary = Path(out_dir) / 'sweep_summary.json'

    # Check if this seed is already complete
    if summary.exists():
        rows = json.loads(summary.read_text())
        n_ok = sum(1 for r in rows if r.get('status') == 'ok')
        # 7 text + 6 multimodal = 13 paired, 7 text all = 20 total runs
        if n_ok >= 20:
            print(f'Seed {seed}: already complete ({n_ok} runs), skipping')
            continue
        else:
            print(f'Seed {seed}: resuming ({n_ok}/20 runs done)')
    else:
        print(f'Seed {seed}: starting fresh')

    print(f'\n{"="*60}')
    print(f'=== SEED {seed} ===')
    print(f'{"="*60}', flush=True)

    result = subprocess.run(
        [sys.executable, '-u', 'scripts/run_matched_sweep.py',
         '--seed', str(seed), '--out', out_dir],
        env=env,
    )

    if result.returncode != 0:
        print(f'!!! Seed {seed} had errors (check logs above)')
    else:
        print(f'--- Seed {seed} complete ---')

# Summary
print(f'\n{"="*60}')
print('=== ALL SEEDS SUMMARY ===')
for seed in SEEDS:
    summary = Path(f'{DRIVE_OUT}/seed-{seed}/sweep_summary.json')
    if summary.exists():
        rows = json.loads(summary.read_text())
        n_ok = sum(1 for r in rows if r.get('status') == 'ok')
        n_fail = sum(1 for r in rows if r.get('status') == 'failed')
        print(f'  Seed {seed}: {n_ok} ok, {n_fail} failed')
    else:
        print(f'  Seed {seed}: not started')

In [ ]:
# Cell 6: Aggregate seeds + download
import os, subprocess, sys, json, tarfile
from pathlib import Path

os.chdir('/content/bangla-multimodal-political-stance')
DRIVE_OUT = '/content/drive/MyDrive/bmpb/seeds'
SEEDS = [42, 43, 44, 45, 46]

# Run seed analysis if the script exists
if Path('scripts/seed_analysis.py').exists():
    subprocess.run(
        [sys.executable, '-u', 'scripts/seed_analysis.py', '--seeds-dir', DRIVE_OUT],
        check=False,
    )

# Print per-seed top-line results
print('\n=== Per-seed macro-F1 (paired_headlines) ===')
for seed in SEEDS:
    summary = Path(f'{DRIVE_OUT}/seed-{seed}/sweep_summary.json')
    if not summary.exists():
        continue
    rows = json.loads(summary.read_text())
    paired = [r for r in rows if r.get('population') == 'paired_headlines' and r.get('status') == 'ok']
    for r in sorted(paired, key=lambda x: x.get('macro_f1', 0), reverse=True):
        print(f"  seed={seed}  {r['name']:20s}  F1={r['macro_f1']:.3f}")

# Archive and download
archive_path = '/content/bmpb-seeds.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(DRIVE_OUT, arcname='seeds')
print(f'\nArchive: {archive_path} ({Path(archive_path).stat().st_size / 1e6:.1f} MB)')

from google.colab import files
files.download(archive_path)